# Micro-Credit Default Prediction using Telecom Behavioral Data

## Project Overview
This project builds a binary classification model to predict whether a mobile subscriber will **default on a micro-credit loan**, using 37 telecom behavioral features such as recharge patterns, loan history, call usage, and more.

**Dataset:** `Micro-credit-Data-file.csv`  
**Target Variable:** `label` (0 = No Default, 1 = Default)  
**Best Model Results:**
- Log Loss: 0.2236
- Precision: 92.3%
- Recall: 98.0%

---

## 1. Install Dependencies

In [ ]:
!pip install scikit-learn xgboost lightgbm catboost optuna

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, precision_score, recall_score, classification_report

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

## 3. Load Data
The dataset contains 37 columns including telecom usage metrics (recharge amounts, call patterns, loan history) and a binary `label` column indicating default status.

In [ ]:
df = pd.read_csv('Micro-credit-Data-file.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 4. Exploratory Data Analysis (EDA)
Checking class distribution and feature correlations with the target variable.

In [ ]:
sns.countplot(x='label', data=df)
plt.title('Class Distribution (0 = No Default, 1 = Default)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

correlations = df.corr()['label'].sort_values(ascending=False)
print('Top 10 features correlated with default:')
print(correlations.head(10))

## 5. Data Preprocessing
- Drop non-informative identifier columns (`msisdn`, `pcircle`)
- Parse the `pdate` column and extract month as a numeric feature
- Standardize features using `StandardScaler`
- Stratified train/test split (80/20)

In [ ]:
df = df.drop(['msisdn', 'pcircle'], axis=1)
df['pdate'] = pd.to_datetime(df['pdate'])
df['month'] = df['pdate'].dt.month
df = df.drop('pdate', axis=1)

X = df.drop('label', axis=1)
y = df['label']

feature_names = X.columns.tolist()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, stratify=y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape}, Test set: {X_test.shape}')

## 6. Feature Engineering
Creating a `recharge_intensity30` feature that captures average recharge amount per transaction over 30 days.

In [ ]:
df['recharge_intensity30'] = df['sumamnt_ma_rech30'] / (df['cnt_ma_rech30'] + 1)
print('New feature added: recharge_intensity30')

## 7. Model Training with Hyperparameter Tuning
Using `RandomizedSearchCV` to tune XGBoost over a grid of `max_depth`, `learning_rate`, `n_estimators`, and `subsample`. Optimizing for log loss via 3-fold cross-validation.

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')

params = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1]
}

grid = RandomizedSearchCV(
    xgb,
    param_distributions=params,
    scoring='neg_log_loss',
    cv=3,
    n_iter=10,
    random_state=42
)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print('Best Parameters:', grid.best_params_)

## 8. Model Evaluation
Evaluating the best XGBoost model on the held-out test set.

In [ ]:
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

print('=== Model Performance ===')
print(f'Log Loss : {log_loss(y_test, y_pred_proba):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred))

## 9. Feature Importance
Visualizing the top 10 most influential features as determined by the XGBoost model.

In [ ]:
importances = best_model.feature_importances_
feat_importances = pd.Series(importances, index=feature_names)

feat_importances.nlargest(10).plot(kind='barh', color='steelblue')
plt.title('Top 10 Feature Importances (XGBoost)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()